<a href="https://colab.research.google.com/github/utopienne/scraps/blob/main/GradingCode7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install canvasapi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.8/116.8 kB 3.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
API_TOKEN = "16765~wNL8uQtNm2cFVA83ZLA8CAm78uQeHGJmMHJJtKW9aEL4YFnV9LJ9BXBAQrXaNHNB"
API_URL = "https://umsystem.instructure.com"
COURSE_ID = 415398
ASSIGNMENT_ID = 3538407
#week 4
#SECTION_ID = 502667
drive.mount('/content/drive', force_remount=True) # Force remount to ensure it's fresh
# Adjust this path if your folder is named differently (e.g., 'MyDrive' vs 'My Drive')
SAVE_PATH = '/content/drive/MyDrive/Grading/Week4_Export'

Mounted at /content/drive


In [ ]:
#!pip install canvasapi
import os
from canvasapi import Canvas
from google.colab import drive

# --- CONFIGURATION (Directly from your details) ---
RAW_TOKEN = "16765~wNL8uQtNm2cFVA83ZLA8CAm78uQeHGJmMHJJtKW9aEL4YFnV9LJ9BXBAQrXaNHNB"

tokenMCC = "20979~EMB2vfEfWNMVEhzAFkzQzrwPJ7TzFDT9XvhzYkZ4fGJYmGyJzDVrhDkwQLB6keMa"
API_URL = "https://umsystem.instructure.com"
#COURSE_ID = 382797
#ASSIGNMENT_ID = 3538527

# Clean the token of any invisible formatting or spaces
CLEAN_TOKEN = RAW_TOKEN.strip()

# Initialize with the cleaned token
canvas = Canvas(API_URL, CLEAN_TOKEN)

print("Starting UMSystem Handshake...")

try:
    # Test 1: Identify the User
    user = canvas.get_current_user()
    print(f"✅ Token Verified! Authenticated as: {user.name}")

    # Test 2: Identify the Course
    course = canvas.get_course(COURSE_ID)
    print(f"✅ Course Found: {course.name}")

    # Test 3: Identify the Assignment
    assignment = course.get_assignment(ASSIGNMENT_ID)
    print(f"✅ Assignment Found: {assignment.name}")

    print("\n--- ALL SYSTEMS GO ---")

except Exception as e:
    print("\n❌ HANDSHAKE FAILED")
    print(f"Error Type: {type(e).__name__}")
    print(f"Details: {e}")
    print("\nTroubleshooting Tip: If you just generated the token, sometimes it takes 60 seconds to propagate through UMSystem's SSO.")

Starting UMSystem Handshake...

❌ HANDSHAKE FAILED
Error Type: InvalidAccessToken
Details: [{'message': 'Expired access token.', 'expired_at': '2026-05-31T05:00:00Z'}]

Troubleshooting Tip: If you just generated the token, sometimes it takes 60 seconds to propagate through UMSystem's SSO.


In [ ]:
##Download and extract (ungraded) assignments
import os
import io
import zipfile
import requests
from canvasapi import Canvas
from google.colab import drive
import time

# --- 1. MOUNT STORAGE ---
drive.mount('/content/drive', force_remount=True) # Force remount to ensure it's fresh
# Adjust this path if your folder is named differently (e.g., 'MyDrive' vs 'My Drive')
SAVE_PATH = '/content/drive/MyDrive/Grading130/Week4_Drafting/GradingFolder4'
#"C:\Users\gypsy\My Drive (utopienne@gmail.com)\Grading130\Week4_Drafting\GradingFolder4"
#"C:\Users\gypsy\My Drive (utopienne@gmail.com)\Grading130\Week3_Drafting\Week3_Grading4"



canvas = Canvas(API_URL, API_TOKEN)

# Adding a retry mechanism for API calls that might fail due to transient network issues (like 504)
MAX_RETRIES = 5
RETRY_DELAY_SECONDS = 5

assignment = None
for attempt in range(MAX_RETRIES):
    try:
        assignment = canvas.get_course(COURSE_ID).get_assignment(ASSIGNMENT_ID)
        print(f"🚀 Starting Sync for: {assignment.name}")
        break # If successful, break out of the retry loop
    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {e}")
        if attempt < MAX_RETRIES - 1:
            print(f"Retrying in {RETRY_DELAY_SECONDS} seconds...")
            time.sleep(RETRY_DELAY_SECONDS)
        else:
            print("Max retries reached. Could not retrieve assignment.")
            raise # Re-raise the last exception if all retries fail

# --- 3. PROCESSING WITH "ALREADY DOWNLOADED" CHECK ---
# Include 'grade' to check if student already has a grade
# IMPORTANT: Removed 'enrollments' as the user wants to use df_text for section data
submissions = assignment.get_submissions(include=['user', 'grade'])
count_downloaded = 0
count_skipped = 0

for sub in submissions:
    current_student_name = "Unknown Student"
    try:
        # Modified: Format name as Lastname_Firstname
        full_name_parts = sub.user['name'].split()
        if len(full_name_parts) > 1:
            current_student_name = "_".join(full_name_parts[::-1])
        else:
            current_student_name = full_name_parts[0] if full_name_parts else "Unknown_Student"

        # NEW LOGIC: Check if student already has a grade
        if sub.grade is not None:
            print(f"Skipping {current_student_name}: Already graded (Grade: {sub.grade}).")
            count_skipped += 1 # Count as skipped because files are not downloaded again
            continue

        if sub.workflow_state == 'submitted':
            attachments = getattr(sub, 'attachments', [])

            if not attachments:
                print(f"Skipping {current_student_name}: No attachments found.")
                continue

            for attachment in attachments:
                ext = os.path.splitext(attachment.filename)[1].lower()

                # Case A: Student uploaded a ZIP (Extract only CAD files)
                if ext == '.zip':
                    print(f"📦 Processing ZIP from {current_student_name} ({attachment.filename})...")
                    try:
                        # We still need to download the zip to inspect its contents
                        r = requests.get(attachment.url)
                        r.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)
                    except requests.exceptions.RequestException as req_e:
                        print(f"  ❌ Error downloading ZIP for {current_student_name} ({attachment.filename}): {req_e}")
                        continue # Skip to the next attachment/attachment loop

                    try:
                        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                            for f_name in z.namelist():
                                if f_name.lower().endswith(('.dwg', '.sldprt')):
                                    clean_name = f"{current_student_name}_{os.path.basename(f_name)}"
                                    full_save_path = os.path.join(SAVE_PATH, clean_name)

                                    if os.path.exists(full_save_path):
                                        print(f"  --> Skipping existing file: {clean_name}")
                                        count_skipped += 1
                                        continue # Skip to next file in zip

                                    try:
                                        with z.open(f_name) as source, open(full_save_path, "wb") as target:
                                            target.write(source.read())
                                        print(f"  --> Saved: {clean_name}")
                                        count_downloaded += 1
                                    except Exception as inner_e:
                                        print(f"  ⚠️ Error processing file {f_name} in zip for {current_student_name}: {inner_e}")
                                else:
                                    print(f"  Skipping non-CAD file in zip: {f_name}")
                    except zipfile.BadZipFile as bad_zip_e:
                        print(f"  ❌ Error: ZIP file for {current_student_name} ({attachment.filename}) is corrupted or invalid: {bad_zip_e}")
                    except Exception as general_zip_e:
                        print(f"  ❌ An unexpected error occurred while processing ZIP for {current_student_name} ({attachment.filename}): {general_zip_e}")

                # Case B: Student uploaded a DWG/CAD file directly
                elif ext in ['.dwg', '.sldprt']:
                    clean_name = f"{current_student_name}_{attachment.filename}"
                    full_save_path = os.path.join(SAVE_PATH, clean_name)

                    if os.path.exists(full_save_path):
                        print(f"📄 Skipping existing file: {clean_name}")
                        count_skipped += 1
                        continue # Skip to next attachment

                    print(f"📄 Saving DWG/CAD from {current_student_name} ({attachment.filename})...")
                    try:
                        r = requests.get(attachment.url)
                        r.raise_for_status() # Raise an exception for HTTP errors
                        with open(full_save_path, 'wb') as f:
                            f.write(r.content)
                        print(f"  --> Saved: {clean_name}")
                        count_downloaded += 1
                    except requests.exceptions.RequestException as req_e:
                        print(f"  ❌ Error downloading DWG/CAD for {current_student_name} ({attachment.filename}): {req_e}")
                    except Exception as e:
                        print(f"  ⚠️ Error saving DWG/CAD for {current_student_name} ({attachment.filename}): {e}")
                else:
                    print(f"Skipping {current_student_name}: Attachment '{attachment.filename}' is not a ZIP or CAD file.")
        else:
            print(f"Skipping {current_student_name}: Not submitted.")
    except Exception as e:
        # Catch any other unexpected errors during submission processing
        print(f"⚠️ Error processing submission for {current_student_name}: {e}")
        continue

print(f"\n✅ Sync Complete! Downloaded {count_downloaded} new files and skipped {count_skipped} existing files. All files are in {SAVE_PATH}")

Mounted at /content/drive
🚀 Starting Sync for: Homework 4
Skipping Small_Chauncey: Already graded (Grade: 94).
Skipping Paul_Francis: Already graded (Grade: 83).
Skipping Daghmoush_Mohammed: Already graded (Grade: 55.33).
Skipping Gikonyo_Henry: Already graded (Grade: 92).
Skipping Mussa_Natilina: Already graded (Grade: 57.33).
Skipping Peister_Ellie: Already graded (Grade: 97.33).
📦 Processing ZIP from Orlick_Wendy (AutoCAD+-+1.zip)...
  --> Saved: Orlick_Wendy_HW4_1.dwg
  --> Saved: Orlick_Wendy_HW4_2.dwg
  --> Saved: Orlick_Wendy_HW4_3.dwg
📦 Processing ZIP from Al-Shwbaki_Shahd (HW+4.zip)...
  --> Saved: Al-Shwbaki_Shahd_HW 4-2.dwg
  --> Saved: Al-Shwbaki_Shahd_._HW 4-2.dwg
  --> Saved: Al-Shwbaki_Shahd_HW4-1.dwg
  --> Saved: Al-Shwbaki_Shahd_._HW4-1.dwg
  --> Saved: Al-Shwbaki_Shahd_HW4-3.dwg
  --> Saved: Al-Shwbaki_Shahd_._HW4-3.dwg
📦 Processing ZIP from Arambula_Martinez_Alexia (AnchorBase.zip)...
  --> Saved: Arambula_Martinez_Alexia_AnchorBase.dwg
  --> Saved: Arambula_Martinez

In [ ]:
df_text

,Student Name,Login ID,Section
0,Alonso_Joshua,ja7xm,005
1,Bassembat_Hanva,hhb2cp,005
2,Buseck_Gauge,grbb5c,005
3,Daghmoush_Mohammed,mdg8d,005
4,Diaz_Michael,madkmp,005
...,...,...,...
81,Tomlinson_Troy,ttrcg,002
82,Watkins Davis_Grant,gaww29,002
83,Weichert_Cadence,cw7hn,002
84,Xiloj_Josue,jxhxc,002


### Generating Master Roster from Canvas

In [ ]:
import pandas as pd
import os
import shutil
from google.colab import drive

# 1. Setup
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Grading/'
ROSTER_PATH = os.path.join(BASE_PATH, 'Master_Roster.csv')

# 2. Load the Roster
roster = pd.read_csv(ROSTER_PATH)
# Create a dictionary for quick lookup: { "Jane_Doe": "003" }
section_map = dict(zip(roster['Student Name'], roster['Section'].astype(str)))

print("Sorting files into sections...")

# 3. Process Files
files = [f for f in os.listdir(BASE_PATH) if f.endswith('.dwg')]

for filename in files:
    # Extract student name from filename (assuming 'Student_Name_HW.dwg')
    # This splits by the first underscore and reassembles or takes the prefix
    student_name = filename.split('_')[0] + "_" + filename.split('_')[1]

    if student_name in section_map:
        section = section_map[student_name]
        section_folder = os.path.join(BASE_PATH, f"Section_{section}")

        # Create folder if it doesn't exist
        if not os.path.exists(section_folder):
            os.makedirs(section_folder)

        # Move the file
        shutil.move(os.path.join(BASE_PATH, filename),
                    os.path.join(section_folder, filename))
        print(f"✅ Moved {filename} to Section {section}")
    else:
        print(f"⚠️ Could not find section for: {student_name}")

print("\nSorting Complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sorting files into sections...

Sorting Complete.


In [ ]:
import pandas as pd

# Ensure the SAVE_PATH exists before trying to save the CSV
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# Export the DataFrame to CSV
df.to_csv(ROSTER_PATH, index=False)
print(f"Updated CSV file exported to: {ROSTER_PATH}")

Updated CSV file exported to: /content/drive/MyDrive/Grading/Master_Roster.csv


In [ ]:
import pandas as pd
import os
from canvasapi import Canvas # Import Canvas

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# --- Canvas API Setup to get submissions (Added to resolve NameError) ---
canvas = Canvas(API_URL, API_KEY)
course = canvas.get_course(COURSE_ID)
assignment = course.get_assignment(ASSIGNMENT_ID) # Get the assignment object
print(f"Retrieving submissions for: {assignment.name}")

# --- Fetch sections and enrollments to build section_map from Canvas ---
section_map = {}
print("Fetching section data from Canvas...")
try:
    # include=['students'] fetches a list of student objects within each section
    sections = course.get_sections(include=['students'])
    for section in sections:
        section_name = section.name # e.g., "Section 001" or "001"
        # Extract just the number and format it (e.g., "001")
        section_number = section_name.replace("Section ", "").strip().zfill(3)

        if hasattr(section, 'students'):
            for student_in_section in section.students:
                student_canvas_name = student_in_section['name']

                # Standardize student name for the map key (Lastname_Firstname)
                name_parts = student_canvas_name.split()
                standardized_name_for_map = ""
                if len(name_parts) >= 2:
                    first_name_part = name_parts[0]
                    last_name_part = name_parts[-1]
                    standardized_name_for_map = f"{last_name_part}_{first_name_part}"
                elif len(name_parts) == 1:
                    standardized_name_for_map = name_parts[0]
                else:
                    standardized_name_for_map = "Unknown_Student" # Fallback for odd names

                section_map[standardized_name_for_map] = section_number
    print(f"Successfully fetched {len(section_map)} student-section mappings from Canvas.")
except Exception as e:
    print(f"Error fetching sections from Canvas: {e}")
    print("Proceeding with an empty section map. Sections will be 'Unknown'.")
    section_map = {}

# Retrieve submissions
submissions = assignment.get_submissions(include=['user', 'grade'])

grading_data = []

for sub in submissions:
    canvas_full_name = sub.user['name'] # e.g., "Athena Fry", "Joshua David Alonso"
    name_parts = canvas_full_name.split()
    standardized_name = ""

    # New logic: Extract First Name (first word) and Last Name (last word) from Canvas name
    # to match the 'Lastname_Firstname' format in df_text more reliably.
    if len(name_parts) >= 2:
        first_name_part = name_parts[0]
        last_name_part = name_parts[-1]
        standardized_name = f"{last_name_part}_{first_name_part}"
    elif len(name_parts) == 1:
        standardized_name = name_parts[0] # For single-word names
    else:
        standardized_name = "Unknown_Student"

    # Get section from the pre-existing section_map (derived from Canvas data)
    # Use .get() with a default value 'Unknown' if the student isn't found in the map
    student_section = section_map.get(standardized_name, "Unknown")

    grading_data.append({
        'Student Name': sub.user['name'], # Keep original Canvas name for clarity in the sheet
        'Standardized Name': standardized_name, # Add standardized name for reference/debugging
        'Canvas ID': sub.user['id'],
        'Section': student_section, # Add the section here, now from Canvas data
        'Grade': sub.score if sub.score else "",
        'Comments': ""  # This is where you will type your notes
    })

df = pd.DataFrame(grading_data)

# Reorder columns for better readability
df = df[['Student Name', 'Standardized Name', 'Canvas ID', 'Section', 'Grade', 'Comments']]

df.to_csv('/content/drive/MyDrive/Grading130/Week3_Drafting/Week3_Grading4/Grading_Sheet3.csv', index=False)
print("CSV Created! You can open this in Google Sheets to grade.")

Retrieving submissions for: Homework 3
Fetching section data from Canvas...
Successfully fetched 89 student-section mappings from Canvas.
CSV Created! You can open this in Google Sheets to grade.


In [ ]:
##For Uploading Grades
import pandas as pd
from google.colab import drive
from canvasapi import Canvas
import time

drive.mount('/content/drive', force_remount=True) # Force remount to ensure it's fresh
# Adjust this path if your folder is named differently (e.g., 'MyDrive' vs 'My Drive')
SAVE_PATH = '/content/drive/MyDrive/Grading130/Week4_Drafting'
#"C:\Users\gypsy\My Drive (utopienne@gmail.com)\Grading130\Week4_Drafting\Grading_SheetDone.csv"

# --- Re-initialize Canvas API and Assignment Object ---
API_URL = "https://umsystem.instructure.com"
API_KEY = "16765~wNL8uQtNm2cFVA83ZLA8CAm78uQeHGJmMHJJtKW9aEL4YFnV9LJ9BXBAQrXaNHNB"
COURSE_ID = 382797
#ASSIGNMENT_ID = 3538404
##week 3


canvas = Canvas(API_URL, API_KEY)

# Adding a retry mechanism for API calls that might fail due to transient network issues (like 504)
MAX_RETRIES = 5
RETRY_DELAY_SECONDS = 5

assignment = None
for attempt in range(MAX_RETRIES):
    try:
        assignment = canvas.get_course(COURSE_ID).get_assignment(ASSIGNMENT_ID)
        print(f"🚀 Successfully retrieved Assignment: {assignment.name}")
        break # If successful, break out of the retry loop
    except Exception as e:
        print(f"Attempt {attempt + 1} failed to retrieve assignment: {e}")
        if attempt < MAX_RETRIES - 1:
            print(f"Retrying in {RETRY_DELAY_SECONDS} seconds...")
            time.sleep(RETRY_DELAY_SECONDS)
        else:
            print("Max retries reached. Could not retrieve assignment. Grade upload aborted.")
            raise # Re-raise the last exception if all retries fail

# Load your filled-out sheet from your 1TB Drive
CSV_PATH = '/content/drive/MyDrive/Grading130/Week4_Drafting/Grading_SheetDone.csv' # Changed to use CSV in runtime
df = pd.read_csv(CSV_PATH, encoding='latin-1') # Changed encoding to 'latin-1'

print("Starting Grade Upload...")
for index, row in df.iterrows():

    student_name= row['Student Name']
    canvas_id = row['Canvas ID']
    grade_to_upload = row['Grade'] if pd.notna(row['Grade']) else None
    comment_to_upload = row['Comments'] if pd.notna(row['Comments']) else None

    # Skip if neither grade nor comment is specified in the CSV row
    if grade_to_upload is None and comment_to_upload is None:
        # print(f"Skipping {student_name}: No grade or comment specified.")
        continue

    try:
        # Get the submission object, including existing comments to check for duplicates
        submission = assignment.get_submission(canvas_id, include=['submission_comments', 'rubric_assessment'])

        update_payload = {}
        comment_payload = {}
        changes_made = False

        # 1. Handle Grade Upload
        if grade_to_upload is not None:
            # Canvas returns score as a string or None, convert for comparison
            current_score = float(submission.score) if submission.score is not None else None

            if current_score is None or abs(current_score - float(grade_to_upload)) > 0.001:
                update_payload['posted_grade'] = grade_to_upload
                print(f"  --> Grade change detected for {student_name}: {grade_to_upload}")
                changes_made = True
            # else:
                # print(f"  --> Grade already up-to-date for {student_name}: {grade_to_upload}")


        # 2. Handle Comment Upload (only if new and not already present)
        if comment_to_upload is not None:
            existing_comments_text = [c.get('comment', '') for c in getattr(submission, 'submission_comments', [])]

            if comment_to_upload not in existing_comments_text:
                comment_payload['text_comment'] = comment_to_upload
                print(f"  --> New comment will be added for {student_name}")
                changes_made = True
            # else:
                # print(f"  --> Comment already exists for {student_name}: '{comment_to_upload}'")

        # Execute the update if there's anything to change
        if changes_made:
            submission.edit(submission=update_payload, comment=comment_payload)
            print(f"✅ Uploaded: {student_name}")
        else:
            print(f"☑️ No changes needed for {student_name}")

    except Exception as e:
        print(f"❌ Failed to upload for {student_name}: {e}")

print("\nAll done! Your Canvas gradebook is now up to date.")

Mounted at /content/drive
🚀 Successfully retrieved Assignment: Homework 4
Starting Grade Upload...
  --> Grade change detected for Abdullahi Abdullahi: 91.33333333
  --> New comment will be added for Abdullahi Abdullahi
✅ Uploaded: Abdullahi Abdullahi
  --> Grade change detected for Shahd Al-Shwbaki: 88.0
  --> New comment will be added for Shahd Al-Shwbaki
✅ Uploaded: Shahd Al-Shwbaki
  --> Grade change detected for Hasan Alturkmani: 90.0
  --> New comment will be added for Hasan Alturkmani
✅ Uploaded: Hasan Alturkmani
  --> Grade change detected for Alexia Martinez Arambula: 79.33333333
  --> New comment will be added for Alexia Martinez Arambula
✅ Uploaded: Alexia Martinez Arambula
  --> Grade change detected for Fernando Banuelas: 82.66666667
  --> New comment will be added for Fernando Banuelas
✅ Uploaded: Fernando Banuelas
  --> Grade change detected for Alisha Bibi: 98.0
✅ Uploaded: Alisha Bibi
  --> Grade change detected for Adam Borra: 95.33333333
  --> New comment will be add

In [ ]:
##Install the right package
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.9 MB/s eta 0:00:00


In [ ]:
##Move files over
##This is the Booklet Maker
import os
from PyPDF2 import PdfWriter, PdfReader
from fpdf import FPDF

# --- CONFIGURATION ---
student_folder = r"/content/drive/MyDrive/Grading/Week5_Export"
output_pdf = "Grading_Booklet_Asgn5.pdf"

def calculate_score(audit_path):
    score = 35.0  # Starting total based on your rubric
    deductions = []

    with open(audit_path, "r") as f:
        data = f.read()

    # 1. Precision Check
    if "Precision:2" not in data:
        score -= 3.0
        deductions.append("Wrong Precision (-3 pts)")

    # 2. Template Check
    if "Template:WRONG" in data:
        score -= 5.0
        deductions.append("Wrong Template (-5 pts)")

    # 3. Layer Check (Checking the 8 layers we defined)
    required_layers = ["PART", "CENTER", "HIDDEN", "DIMENSION", "PROJECTION 1"]
    for layer in required_layers:
        if f"{layer}:MISSING" in data:
            score -= 1.0
            deductions.append(f"Missing Layer: {layer} (-1 pt)")

    return score, deductions

def create_master_booklet():
    merger = PdfWriter()

    # Get all students who have an audit file
    audit_files = [f for f in os.listdir(student_folder) if f.endswith("_audit.txt")]

    for audit_file in audit_files:
        student_name = audit_file.replace("_audit.txt", "")
        drawing_pdf = os.path.join(student_folder, f"{student_name}.pdf")
        audit_path = os.path.join(student_folder, audit_file)

        # Calculate Math
        final_score, reasons = calculate_score(audit_path)

        # --- Create a Temporary Cover Page for this student ---
        temp_sheet = FPDF()
        temp_sheet.add_page()
        temp_sheet.set_font("Arial", "B", 16)
        temp_sheet.cell(0, 10, f"Student: {student_name}", ln=True)
        temp_sheet.set_font("Arial", "B", 12)
        temp_sheet.cell(0, 10, f"Calculated Score: {final_score} / 35", ln=True)
        temp_sheet.ln(5)
        temp_sheet.set_font("Arial", "", 10)
        for item in reasons:
            temp_sheet.cell(0, 6, f"- {item}", ln=True)

        temp_sheet_path = "temp_cover.pdf"
        temp_sheet.output(temp_sheet_path)

        # --- Merge: Cover Page + Student Drawing ---
        merger.append(temp_sheet_path)
        if os.path.exists(drawing_pdf):
            merger.append(drawing_pdf)

    # Save the final big file
    with open(output_file, "wb") as f:
        merger.write(f)

    # Cleanup
    if os.path.exists("temp_cover.pdf"): os.remove("temp_cover.pdf")
    print(f"Success! Created {output_file}")

if __name__ == "__main__":
    create_master_booklet()

# Task
Modify cell `1aZQgeKdJdzc` to properly install the `PyPDF2` package.

## Fix SyntaxError for pip install

### Subtask:
Modify cell `1aZQgeKdJdzc` to prefix `pip install PyPDF2` with an exclamation mark `!` to correctly execute it as a shell command in the Colab environment.


## Summary:

### Data Analysis Key Findings
*   A `SyntaxError` occurred when attempting to install the `PyPDF2` package using the command `pip install PyPDF2`.
*   The error was resolved by modifying the command to `!pip install PyPDF2`, prefixing it with an exclamation mark.

### Insights or Next Steps
*   Ensure that shell commands, such as `pip install`, are always prefixed with an exclamation mark `!` when executed directly within a Colab notebook cell to avoid syntax errors and ensure proper execution.
